# 390 — Classification results (read me)

This is the **results page** for `03_FBM_Classifying`. It doesn't compute anything heavy —
it pulls the artifacts that 320 and 330 wrote and lays them out with the story and a guide
to reading each figure. Run 310 → 320 → 330 first, then run this top-to-bottom.

> 📖 **Full reference — definitions, equations, and what to look for for every plot/metric:
> [`READING_GUIDE.md`](READING_GUIDE.md).** The summaries below are the short version.

---

## What was asked

Two supervised questions, both built on the per-electrode ERSPs from `01_FBM_Analysis`:

1. **Condition decoding** — from one electrode's spectro-temporal response, can we tell the
   trial was **audio**, **picture**, or **reading**? *(3 classes; one sample per
   high-activity electrode × condition.)*
2. **Parcellation decoding** — from an electrode's profile **across all three conditions
   concatenated**, can we tell which **Yeo network** it sits in? *(7- or 17-class; one
   sample per electrode.)*

## The comparison is built to be *fair* — matched, nested feature variants

The point is to ask **"does the full spectrum beat high-gamma alone?"** For that to mean
anything, HG and full-spectrum must differ **only** in frequency content — same time grid,
same electrodes, same pipeline. So the four variants form two **matched pairs**:

| pair | full spectrum | high-gamma | what it isolates |
|---|---|---|---|
| 300-time | `full_300` (15 bands × 300) | `hg_300` (1 line × 300) | frequency content, time fixed |
| 30-time | `full_30` (15 bands × 30) | `hg_30` (1 line × 30) | frequency content, time fixed |

Read **`full_* vs hg_*` at the *same* time length** — never `hg_300` (fine time) against
`full_30` (coarse time), which is what an earlier version did and which tangles frequency
with time. `full_300 vs full_30` (or `hg_300 vs hg_30`) isolates the time-resolution effect.

> ⚠️ **Frequency and time go hand in hand.** The STFT has a fixed time–frequency resolution
> trade-off, so HG-with-fine-time vs full-with-coarse-time can never be a *perfectly* clean
> separation. Matching the time grid removes the gross confound; the residual coupling is a
> genuine caveat — treat differences between the 300 and 30 families as suggestive, not proof.

Two classifiers per variant: **logistic regression** (linear, interpretable) and **random
forest** (non-linear).

## The amplitude confound — why we also run row-normed and discretized variants

A strong visually-evoked electrode is "high across many band×time cells at once," so after
column z-scoring its feature vector still has a large norm pointing in a consistent
direction. Both linear models and trees can then separate classes on "how strongly does
this electrode respond" instead of "what shape is its response."

That's especially corrosive for your two tasks:

- **Condition decoding:** picture/reading evoke far larger responses than audio, so the
  model can score well on raw amplitude and tell you nothing about whether the
  spectro-temporal pattern differs.
- **Parcellation:** sensory networks (visual/auditory cortex) are simply louder, so "which
  Yeo network" partly collapses to "how strong is this site."

Per-feature normalization (`StandardScaler`) **cannot** fix this — it rescales each *column*
across electrodes, but the confound lives in the *row* (the sample's overall magnitude). So
for the full-spectrum grid we run an **amplitude triad** at each time resolution:

| representation | variant | amplitude | keeps |
|---|---|---|---|
| **continuous** | `full_300`, `full_30` | intact | everything (incl. magnitude) |
| **row-normalised** | `full_300_rn`, `full_30_rn` | overall magnitude removed (each sample unit-L2) | graded *shape* |
| **discretized** | `m101_300`, `m101_30` | fully discarded | only **sign** of *score-gated* significant blobs (−1/0/+1) |

**The discretized map is score-gated on purpose:** only blobs whose significance score
clears the `M101_SCORE_PCT` percentile are painted ±1 — low-score / weak / noisy segments
stay 0, so we never discretize noise into a confident sign.

**Reading the triad:** if `full ≈ rn ≈ m101`, the *pattern* carries the signal and amplitude
wasn't doing the work (reassuring). If `full ≫ rn ≳ m101`, a lot of the continuous result
was the amplitude confound — the row-normed / discretized numbers are the honest ones. If
`m101 > full`, amplitude was actually a distraction and sign is cleaner. (Caveat: if the
real signal is genuinely *graded*, `m101` will underperform — that's an answer, not a bug.)

## How it was validated — nested GroupKFold by patient

Patients are the unit of splitting. The **outer** loop holds out whole patients as the test
set; the **inner** loop (on the remaining patients) tunes hyper-parameters. No patient is ever
in both train and test, so every number estimates **generalisation to a new patient**. Feature
scaling is fit on training folds only. Class imbalance is handled with `class_weight='balanced'`,
which is why **balanced accuracy** (not raw accuracy) is the headline.

## How to read each output

| Output | What it is | How to read it |
|---|---|---|
| **Balanced accuracy** | mean per-class recall, out-of-fold | compare to **chance = 1/#classes**; the 95% CI is bootstrapped over patients |
| **Permutation p** | balanced accuracy vs a patient-shuffled-label null | `p < 0.05` ⇒ separability is above chance, not luck |
| **Confusion matrix** | row-normalised: rows = true class, cols = predicted | bright diagonal = clean separation; bright off-diagonal = systematic confusion between two classes |
| **Per-class strength** | recall ± bootstrap CI per class | bars above the dashed chance line with `*` (FDR-corrected) are the classes that genuinely separate; `ns` = not distinguishable |
| **Feature importance** | what drove the model | which **condition** (Task B), **frequency band**, and **time bin** carried the signal |
| **Class × feature heatmap** | rows = classes (regions / conditions), cols = frequency bands × condition; colour = per-class mean response | read **down a column** to see which classes light up in a band; **across a row** to see a class's spectral fingerprint. Red = above-average, blue = below |
| **Per-class ERSP profile** | the **full-spectrum** ERSP of each class, **concatenated across audio+picture+reading** (a spectrogram triptych — *not* a single line) | this is the real response signature; for parcellation each Yeo network gets `[audio｜picture｜reading]` side by side. **Always full-spectrum**, even on HG runs, so you see the whole map HG is a slice of |

**Reading the ERSP profile — the stim→response structure.** Each ERSP is time-warped so the
**first half is the stimulus** (viewing / hearing) and the **second half is the response**.
The **dashed grey line at 50%** of every condition block marks that boundary: left of it =
sensing, right of it = response/production. So "an early-vs-late patch" literally means
"during perception vs during the response."
| **Coefficient heatmap** (LR) | rows = classes, cols = bands; colour = signed linear weight | what the model *uses* (vs the raw profile above): which band pushes an electrode **toward** (red) or **away from** (blue) each class |

`*** p<.001 · ** p<.01 · * p<.05 · ns = not significant` (FDR-corrected across classes).


In [ ]:
import sys
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

def show(md):
    display(Markdown(md))

def show_run(task, variant, classifier):
    r = C.load_run(task, variant, classifier)
    if r is None:
        show(f'> _no run found for **{task} · {variant} · {classifier}** — run 320/330 first_')
        return
    o = r['metrics']['overall']
    p = o.get('permutation_p')
    ci = o.get('balanced_accuracy_ci', [float('nan'), float('nan')])
    show(f"### {task} · {C.VARIANT_LABELS[variant]} · {C.CLASSIFIER_LABELS[classifier]}\n"
         f"- balanced accuracy **{o['balanced_accuracy']:.3f}** "
         f"(chance {o['chance_level']:.3f}; 95% CI {ci[0]:.3f}–{ci[1]:.3f})\n"
         f"- macro-F1 {o['macro_f1']:.3f} · accuracy {o['accuracy']:.3f}"
         + (f" · **permutation p = {p:.4f}**" if p is not None else "")
         + f" · n={o['n_samples']} over {o['n_patients']} patients")
    for stem in ('confusion_matrix', 'per_class_strength', 'class_ersp_profile', 'coef_ersp_maps',
                 'class_feature_heatmap', 'coef_heatmap', 'permutation_null',
                 'feature_importance'):
        if stem in r['figures']:
            display(Image(filename=str(r['figures'][stem])))
    cols = [c for c in ['class', 'support', 'recall', 'recall_ci_lo', 'recall_ci_hi',
                        'precision', 'f1', 'roc_auc_ovr', 'perm_p_fdr']
            if c in r['per_class'].columns]
    display(r['per_class'][cols].round(3))


## All experiments at a glance
One row per run. Sort/scan the `balanced_accuracy` vs `chance_level` gap and the
`permutation_p` column to see which representations decode best.


In [ ]:
runs = C.list_runs()
if not len(runs):
    show('> _no runs yet — execute 320 and 330 first._')
else:
    display(runs[['task', 'variant', 'classifier', 'balanced_accuracy', 'chance_level',
                  'macro_f1', 'permutation_p']].round(4))
    r2 = runs.copy()
    r2['exp'] = r2.task + ' · ' + r2.variant + ' · ' + r2.classifier
    r2 = r2.sort_values('balanced_accuracy')
    fig, ax = plt.subplots(figsize=(9, 0.38 * len(r2) + 1))
    ax.barh(r2['exp'], r2['balanced_accuracy'], color='#4363d8', alpha=0.85)
    for i, (ba, ch) in enumerate(zip(r2.balanced_accuracy, r2.chance_level)):
        ax.plot([ch, ch], [i - 0.45, i + 0.45], color='#cc0033', lw=2)
    ax.set_xlabel('balanced accuracy   (red tick = chance)')
    ax.set_title('Separability across all classification experiments')
    plt.tight_layout(); plt.show()


## Amplitude triad — is the decoding pattern or just power?

For each task / classifier / time grid, compare the full-spectrum representation against its
**row-normalised** and **discretized** counterparts. Within a row the task (and therefore
chance) is fixed, so the three numbers are directly comparable:
**continuous ≈ row-normed ≈ discretized** → pattern carries it; **continuous ≫ the other two**
→ the result was riding on response *amplitude* (the strong-visual-electrode confound).


In [ ]:
runs = C.list_runs()
if not len(runs):
    show('> _no runs yet._')
else:
    triad = {'full_300': ('continuous', 300), 'full_30': ('continuous', 30),
             'full_300_rn': ('row-normed', 300), 'full_30_rn': ('row-normed', 30),
             'm101_300': ('discretized', 300), 'm101_30': ('discretized', 30)}
    t = runs[runs.variant.isin(triad)].copy()
    if len(t):
        t['rep'] = t.variant.map(lambda v: triad[v][0])
        t['time'] = t.variant.map(lambda v: triad[v][1])
        piv = t.pivot_table(index=['task', 'classifier', 'time'], columns='rep',
                            values='balanced_accuracy')
        piv = piv.reindex(columns=['continuous', 'row-normed', 'discretized'])
        display(piv.round(3))
    else:
        show('> _amplitude-triad variants not run yet._')


# Task A — Condition decoding (audio / picture / reading)

Chance = 0.333. A high diagonal here means the spectro-temporal response itself carries
*which modality* the electrode was engaged by. Watch the confusion matrix for the
audio↔reading vs picture structure, and the feature-importance panel for whether high-gamma
or lower bands carry the signal. **Cross-check the amplitude triad above** — if `picture`/
`reading` only separate in the continuous variant, the model was decoding loudness.


In [ ]:
for v in C.VARIANTS:
    for clf in ('logreg', 'rf'):
        show_run('condition', v, clf
                )


# Task B — Parcellation decoding (Yeo networks)

Chance = 1/7 ≈ 0.143 (Yeo-7) and 1/17 ≈ 0.059 (Yeo-17). Because classes are imbalanced, read
**per-class strength** carefully: some networks (with many electrodes, broad coverage) will
separate strongly while sparse networks stay at chance. The feature-importance **by-condition**
panel answers "does one condition carry most of the anatomical signal, or all three jointly?".


In [ ]:
for n_net in (7, 17):
    show(f'## Yeo-{n_net}')
    for v in C.VARIANTS:
        for clf in ('logreg', 'rf'):
            show_run(f'parcellation_yeo{n_net}', v, clf)


# Takeaways & caveats

- **Above-chance ≠ strong.** A run can clear the permutation null yet still confuse most
  classes — always cross-check balanced accuracy, the confusion matrix, and the per-class
  stars together.
- **Per-class is where the story is.** The summary number averages over classes; the
  per-class strength panel shows *which* conditions/networks are actually recoverable and
  which collapse into their neighbours.
- **Patient-grouped CV is conservative.** Numbers are lower than a naive random split would
  give, but they are the honest estimate of decoding in an unseen patient.
- **Feature importance is associational**, not causal — it tells you which bands/time/condition
  the model leaned on, given everything else it had.

_To refresh this page: re-run 320 / 330 (writes new runs), then re-run this notebook — it
always loads the latest run per (task · variant · classifier)._
